In [15]:
from datetime import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [16]:
# using the web archive to get the data becaue the original website is blocked from sending requests.

url = (
    "https://web.archive.org/web/20211224175426id_/"
    "https://www.ccny.cuny.edu/registrar/fall"
)

In [17]:
r = requests.get(url)
html_doc = r.text
soup = BeautifulSoup(html_doc, "html.parser")

In [18]:
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# ">
 <head>
  <meta charset="utf-8"/>
  <script type="text/javascript">
   (window.NREUM||(NREUM={})).init={ajax:{deny_list:["bam.nr-data.net"]}};(window.NREUM||(NREUM={})).loader_config={licenseKey:"NRJS-83581f4fb59b51cbeb7",applicationID:"254978615"};window.NREUM||(NREUM={}),__nr_require=function(t,e,n){function r(n){if(!e[n]){var i=e[n]={exports:{}};t[n][0].call(i.exports,function(e){var i=t[n][1][e];return r(i||e)},i,i.exports)}return e[n].exports}if("function"==typeof __nr_require)return __nr_require;for(var i=0;i<n.length;i++)r(n[i]);return r}({1:[function(t,e,n){function r()

In [20]:

dates = []
dows = []
texts = []

# each calendar row is a <tr> with three <td> cells: date, day, description
for row in soup.find_all("tr"):
    cells = row.find_all("td")
    if len(cells) != 3:
        continue

    date_text = " ".join(cells[0].get_text().split())
    dow_text = " ".join(cells[1].get_text().split())
    text = " ".join(cells[2].get_text(" ").split())

    # "January 1, 2022" has its own year; everything else is 2021
    if "," in date_text:
        date_text, year = date_text.split(",")
    else:
        year = "2021"

    # for ranges like "November 25 - 28", keep the first day
    first_day = date_text.split("-")[0].strip()

    try:
        d = datetime.strptime(first_day + " " + year.strip(), "%B %d %Y").date()
    except ValueError:
        continue  # skips header rows like "Date | Day | Description"

    dates.append(d)
    dows.append(dow_text)
    texts.append(text)


In [21]:
# build the DataFrame from a dict of lists, then make "date" the index
calendar = pd.DataFrame({"date": dates, "dow": dows, "text": texts})
calendar = calendar.set_index("date").sort_index()

In [22]:
print(calendar.head(4))

                  dow                                               text
date                                                                    
2021-08-01     Sunday  Application for degree for January and Februar...
2021-08-18  Wednesday                 Last day to apply for Study Abroad
2021-08-24    Tuesday  Last day of Registration; Last day to file ePe...
2021-08-25  Wednesday  Start of Fall Term; Classes begin; Initial Reg...


In [25]:
print(calendar.tail(4))

                 dow                                              text
date                                                                  
2021-12-27    Monday                                    College Closed
2021-12-28   Tuesday     Final Grade Submission Deadline for Fall 2021
2021-12-31    Friday                                    College Closed
2022-01-01  Saturday  College Closed; Fall 2021 Degree Conferral Date;


In [23]:
print(calendar.info())

<class 'pandas.DataFrame'>
Index: 36 entries, 2021-08-01 to 2022-01-01
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   dow     36 non-null     str  
 1   text    36 non-null     str  
dtypes: str(2)
memory usage: 864.0+ bytes
None
